# Calling the Databricks budgets REST API: three read-only approaches

This notebook calls the account-level **Budgets** API, `GET /api/2.1/accounts/{account_id}/budgets` ([Budgets: list](https://docs.databricks.com/api/account/budgets/list)), and includes a workspace control call. Budgets exist only at the account level; there's no documented workspace-host budgets route, so the notebook doesn't try one.

The approach that works needs an **account-level credential**: a token issued by the account console (`https://accounts.cloud.databricks.com`), not by a workspace. The three approaches all use that one credential and differ only in transport, so they aren't separate authentication experiments.

| # | Approach | Transport | Credential |
|---|----------|-----------|------------|
| Control | `GET /api/2.0/preview/scim/v2/Me` on the workspace | `requests` | Workspace credential |
| 1 | `GET /api/2.1/accounts/{account_id}/budgets` | `requests` with headers from the SDK `Config` | Account-level credential |
| 2 | `AccountClient.budgets.list()` | Databricks SDK for Python | Same account-level credential |
| 3 | `databricks account budgets list` and `databricks api get` | Databricks CLI (local only) | Same account-level profile |
| Diagnostic | Approach 1's request sent with the workspace token | `requests` | Workspace credential, for comparison only |

The [OAuth U2M docs](https://docs.databricks.com/aws/en/dev-tools/auth/oauth-u2m) say workspace-level tokens can call only that workspace's REST APIs, and account APIs need a login against the account console. The diagnostic cell at the end shows what happens if you skip that.

Every call is a `GET` with a finite timeout. Output shows status codes, error codes and counts only. It never shows tokens, headers, the account ID, or budget or principal details.

## Saved outputs in this copy

The outputs below are from a local Jupyter run on 2026-10-19, using workspace profile `DEFAULT` for the control and an account-level U2M profile selected with `DATABRICKS_ACCOUNT_PROFILE`:

- **Control:** HTTP 200.
- **Approach 1 (raw REST):** HTTP 200, 22 budgets across 1 page(s).
- **Approach 2 (SDK):** success, 22 budgets.
- **Approach 3 (CLI):** `account budgets list` exit 0 with 22 budgets; `api get` exit 0 with 22 budgets on one page.
- **Diagnostic (workspace token on the account host):** HTTP 400 `Unable to load OAuth Config`. This is an authentication or configuration failure, and the response doesn't say why.

Budget counts reflect the account at run time and will change. Re-run the notebook to replace these outputs with your own.

## Running locally versus in Databricks

The account-level credential comes from a different place in each environment. The workspace credential is never used as a fallback for the account calls, and the notebook never starts an interactive login.

**Local Jupyter.**
- `DATABRICKS_WORKSPACE_PROFILE` selects the workspace profile for the control and diagnostic calls. The default is `DEFAULT`.
- `DATABRICKS_ACCOUNT_PROFILE` must name an existing account-level profile, meaning one with `host = https://accounts.cloud.databricks.com` and an `account_id`. Create it once in a terminal with `databricks auth login --host https://accounts.cloud.databricks.com --account-id <account-id> --profile <name>`.
- If `DATABRICKS_ACCOUNT_PROFILE` is unset, the account cells report "Not executed".
- `DATABRICKS_ACCOUNT_ID` overrides the profile's `account_id`. Nothing in the notebook hard-codes an account ID.

**Databricks notebook.**
- The control and diagnostic calls use the notebook's own workspace credential. No setup is needed.
- That credential can't make the account calls, and there's no CLI profile on the cluster.
- Instead, put an account-level access token in a secret scope, then set the `ACCOUNT_TOKEN_SECRET_SCOPE`, `ACCOUNT_TOKEN_SECRET_KEY` and `DATABRICKS_ACCOUNT_ID` widgets. The notebook reads the token with `dbutils.secrets.get` and passes it to the SDK `Config` as a bearer token.
- U2M access tokens are short-lived, so the secret needs refreshing. A service principal with OAuth M2M is the durable alternative, but that's a different identity and isn't covered here.

Install requirements locally with `pip install databricks-sdk requests`. Both are preinstalled in Databricks Runtime.

## Configuration and helpers

In Databricks, settings come from widgets first, then environment variables, then defaults. Locally, they come from environment variables, then defaults. The helpers mask emails and hosts, and remove anything that looks like a bearer token, JWT, PAT, email or UUID from error text before printing.

In [1]:
import json
import os
import re
import shutil
import subprocess
from urllib.parse import urlparse

import requests
from databricks.sdk import AccountClient, WorkspaceClient
from databricks.sdk.core import Config

IN_DATABRICKS_RUNTIME = "DATABRICKS_RUNTIME_VERSION" in os.environ


def setting(name, default=""):
    if IN_DATABRICKS_RUNTIME:
        dbutils.widgets.text(name, default)
        value = dbutils.widgets.get(name).strip()
        if value:
            return value
    return os.environ.get(name, default).strip()


WORKSPACE_PROFILE = "" if IN_DATABRICKS_RUNTIME else setting("DATABRICKS_WORKSPACE_PROFILE", "DEFAULT")
ACCOUNT_PROFILE = "" if IN_DATABRICKS_RUNTIME else setting("DATABRICKS_ACCOUNT_PROFILE")
ACCOUNT_TOKEN_SECRET_SCOPE = setting("ACCOUNT_TOKEN_SECRET_SCOPE") if IN_DATABRICKS_RUNTIME else ""
ACCOUNT_TOKEN_SECRET_KEY = setting("ACCOUNT_TOKEN_SECRET_KEY") if IN_DATABRICKS_RUNTIME else ""
ACCOUNT_ID_OVERRIDE = setting("DATABRICKS_ACCOUNT_ID")
ACCOUNTS_HOST = setting("DATABRICKS_ACCOUNTS_HOST", "https://accounts.cloud.databricks.com").rstrip("/")
HTTP_TIMEOUT_SECONDS = int(setting("HTTP_TIMEOUT_SECONDS", "30"))
TIMEOUTS = {"http_timeout_seconds": HTTP_TIMEOUT_SECONDS, "retry_timeout_seconds": HTTP_TIMEOUT_SECONDS * 2}
MAX_BUDGETS_TO_COUNT = 500
MAX_PAGES = 20

SECRET_PATTERNS = [
    re.compile(r"(?i)bearer\s+\S+"),
    re.compile(r"\beyJ[\w-]+\.[\w-]+\.[\w-]*"),
    re.compile(r"\bdap[a-z]?[0-9a-f]{20,}\S*"),
    re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+"),
    re.compile(r"\b[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}\b"),
]


def scrub(value, limit=300):
    text = json.dumps(value, default=str) if isinstance(value, (dict, list, tuple)) else str(value)
    for pattern in SECRET_PATTERNS:
        text = pattern.sub("<redacted>", text)
    return text if len(text) <= limit else text[:limit] + "..."


def mask_email(value):
    value = str(value or "")
    if "@" not in value:
        return "<redacted>" if value else "<unset>"
    name, domain = value.split("@", 1)
    return f"{name[:1]}***@{domain}"


def mask_host(url):
    host = urlparse(url or "").hostname or ""
    if host == urlparse(ACCOUNTS_HOST).hostname:
        return host
    labels = host.split(".")
    return "*." + ".".join(labels[-3:]) if len(labels) > 3 else host


def show(result):
    print(json.dumps(result, indent=2, default=str))


def summarize_http(label, response, list_key=None):
    result = {"call": label, "http_status": response.status_code}
    try:
        body = response.json()
    except ValueError:
        body = {}
        result["body_excerpt"] = scrub(response.text)
    if isinstance(body, dict):
        if body.get("error_code") or body.get("message") or body.get("error"):
            result["error_code"] = scrub(body.get("error_code") or body.get("error") or "", limit=80)
            result["message"] = scrub(body.get("message") or body.get("error_description") or "")
        if list_key and response.ok:
            result[f"{list_key}_returned"] = len(body.get(list_key) or [])
            result["has_next_page"] = bool(body.get("next_page_token"))
    return result


def describe_error(exc):
    return {
        "result": "error",
        "error_type": type(exc).__name__,
        "error_code": scrub(getattr(exc, "error_code", None) or "", limit=80),
        "message": scrub(exc),
    }


w = WorkspaceClient(config=Config(profile=WORKSPACE_PROFILE, **TIMEOUTS) if WORKSPACE_PROFILE else Config(**TIMEOUTS))

show({
    "runtime": "databricks" if IN_DATABRICKS_RUNTIME else "local",
    "workspace_profile": WORKSPACE_PROFILE or "<notebook default auth>",
    "workspace_auth_type": w.config.auth_type,
    "workspace_host": mask_host(w.config.host),
    "account_profile": ACCOUNT_PROFILE or "<unset>",
    "account_token_secret": "set" if ACCOUNT_TOKEN_SECRET_SCOPE and ACCOUNT_TOKEN_SECRET_KEY else "<unset>",
    "account_id_override": "set" if ACCOUNT_ID_OVERRIDE else "<unset>",
    "timeout_seconds": HTTP_TIMEOUT_SECONDS,
})

{
  "runtime": "local",
  "workspace_profile": "DEFAULT",
  "workspace_auth_type": "databricks-cli",
  "workspace_host": "*.cloud.databricks.com",
  "account_profile": "budget-account-test",
  "account_token_secret": "<unset>",
  "account_id_override": "<unset>",
  "timeout_seconds": 30
}


## Control: a workspace API call that should succeed

`GET /api/2.0/preview/scim/v2/Me` ([docs](https://docs.databricks.com/api/workspace/currentuser/me)) returns the calling user and works for any workspace user. It confirms that the workspace credential and network path are fine before the account calls.

`w.config.authenticate()` returns ready-to-send auth headers for whichever credential the SDK resolved. The headers are never printed.

In [2]:
workspace_headers = w.config.authenticate()
me_response = requests.get(
    f"{w.config.host}/api/2.0/preview/scim/v2/Me",
    headers=workspace_headers,
    timeout=HTTP_TIMEOUT_SECONDS,
)
control = summarize_http("GET /api/2.0/preview/scim/v2/Me (workspace)", me_response)
if me_response.ok:
    me = me_response.json()
    group_names = {group.get("display") for group in me.get("groups", [])}
    control.update({
        "user": mask_email(me.get("userName")),
        "active": me.get("active"),
        "group_count": len(group_names),
        "in_workspace_admins_group": "admins" in group_names,
    })
show(control)

{
  "call": "GET /api/2.0/preview/scim/v2/Me (workspace)",
  "http_status": 200,
  "user": "j***@databricks.com",
  "active": true,
  "group_count": 4,
  "in_workspace_admins_group": true
}


## Account-level credential

This cell builds one SDK `Config` for the account host. Approaches 1 to 3 all use it.
- **Locally,** it loads the profile named by `DATABRICKS_ACCOUNT_PROFILE`. The SDK's `databricks-cli` auth type reads the cached U2M token from that profile's earlier `databricks auth login`.
- **In Databricks,** it reads a token from the configured secret.

The cell refuses to continue if the resolved host isn't the account console or no account ID is available. That stops an environment variable such as `DATABRICKS_HOST` from quietly turning the account calls into workspace calls.

**Identity:** the notebook doesn't verify who the account credential belongs to. The token may not carry an email claim, and account-level SCIM can be restricted to account admins. Whether it's the same person as the workspace login depends on who completed the browser login for the account profile.

In [3]:
def account_config():
    if IN_DATABRICKS_RUNTIME:
        if not (ACCOUNT_TOKEN_SECRET_SCOPE and ACCOUNT_TOKEN_SECRET_KEY):
            return None, "set the ACCOUNT_TOKEN_SECRET_SCOPE and ACCOUNT_TOKEN_SECRET_KEY widgets"
        if not ACCOUNT_ID_OVERRIDE:
            return None, "set the DATABRICKS_ACCOUNT_ID widget"
        token = dbutils.secrets.get(scope=ACCOUNT_TOKEN_SECRET_SCOPE, key=ACCOUNT_TOKEN_SECRET_KEY)
        return Config(host=ACCOUNTS_HOST, account_id=ACCOUNT_ID_OVERRIDE, token=token, auth_type="pat", **TIMEOUTS), "secret scope"
    if not ACCOUNT_PROFILE:
        return None, "set DATABRICKS_ACCOUNT_PROFILE to an existing account-level CLI profile"
    overrides = {"account_id": ACCOUNT_ID_OVERRIDE} if ACCOUNT_ID_OVERRIDE else {}
    return Config(profile=ACCOUNT_PROFILE, **overrides, **TIMEOUTS), f"profile {ACCOUNT_PROFILE}"


account_status = {}
try:
    account_cfg, account_source = account_config()
except Exception as exc:
    account_cfg, account_source = None, "account credential could not be loaded"
    account_status.update(describe_error(exc))
if account_cfg is not None:
    problems = []
    if urlparse(account_cfg.host).hostname != urlparse(ACCOUNTS_HOST).hostname:
        problems.append("resolved host is not the accounts host")
    if not account_cfg.account_id:
        problems.append("no account_id from DATABRICKS_ACCOUNT_ID or the account profile")
    if problems:
        account_cfg, account_source = None, "; ".join(problems)
if account_cfg is None:
    account_status.setdefault("result", f"Not executed: {account_source}")
    BUDGETS_PATH = ""
else:
    BUDGETS_PATH = f"/api/2.1/accounts/{account_cfg.account_id}/budgets"
    account_status.update({
        "result": "configured",
        "credential_source": account_source,
        "host": mask_host(account_cfg.host),
        "auth_type": account_cfg.auth_type,
        "account_id": "set from " + ("DATABRICKS_ACCOUNT_ID" if ACCOUNT_ID_OVERRIDE else "account config"),
    })
show(account_status)

{
  "result": "configured",
  "credential_source": "profile budget-account-test",
  "host": "accounts.cloud.databricks.com",
  "auth_type": "databricks-cli",
  "account_id": "set from account config"
}


## Approach 1: raw REST with the account-level credential

`account_cfg.authenticate()` returns auth headers for the account credential, and `requests` sends them to the documented path. The cell follows `next_page_token` for up to `MAX_PAGES` pages and reports only page and budget counts. Use this approach when you want the raw JSON or need to call an endpoint the SDK doesn't cover yet.

In [4]:
approach_1 = {"call": "GET /api/2.1/accounts/{account_id}/budgets (account credential)"}
if not BUDGETS_PATH:
    approach_1["result"] = "Not executed: no account-level credential configured."
else:
    try:
        account_headers = account_cfg.authenticate()
        params, pages, budget_count = {}, 0, 0
        while pages < MAX_PAGES:
            response = requests.get(f"{ACCOUNTS_HOST}{BUDGETS_PATH}", headers=account_headers, params=params, timeout=HTTP_TIMEOUT_SECONDS)
            pages += 1
            if not response.ok:
                approach_1.update(summarize_http(approach_1["call"], response))
                break
            body = response.json()
            budget_count += len(body.get("budgets") or [])
            if not body.get("next_page_token"):
                break
            params = {"page_token": body["next_page_token"]}
        approach_1.update({"http_status": response.status_code, "pages_fetched": pages, "budgets_counted": budget_count})
        if response.ok and body.get("next_page_token"):
            approach_1["stopped_at_max_pages"] = True
    except Exception as exc:
        approach_1.update(describe_error(exc))
show(approach_1)

{
  "call": "GET /api/2.1/accounts/{account_id}/budgets (account credential)",
  "http_status": 200,
  "pages_fetched": 1,
  "budgets_counted": 22
}


## Approach 2: Databricks SDK `AccountClient`

This is the same credential as approach 1. `AccountClient.budgets.list()` calls the same endpoint and handles pagination itself. Use this approach when you want typed objects instead of raw JSON.

In [5]:
approach_2 = {"call": "AccountClient.budgets.list()"}
if account_cfg is None:
    approach_2["result"] = "Not executed: no account-level credential configured."
else:
    try:
        a = AccountClient(config=account_cfg)
        counted = 0
        for _ in a.budgets.list():
            counted += 1
            if counted >= MAX_BUDGETS_TO_COUNT:
                break
        approach_2.update({"result": "success", "budgets_counted": counted})
    except Exception as exc:
        approach_2.update(describe_error(exc))
show(approach_2)

{
  "call": "AccountClient.budgets.list()",
  "result": "success",
  "budgets_counted": 22
}


## Approach 3: Databricks CLI with the same account profile

`databricks account budgets list` is the typed CLI command. `databricks api get` makes the raw REST call to the same documented path. Both run with `--profile` set to the account profile from approach 1, so this is the same credential through a third transport.

This approach only runs locally, because Databricks clusters don't have the CLI or a CLI profile. The cell prints exit codes and counts, never the command's raw output.

In [6]:
def run_cli(label, args):
    result = {"command": label}
    try:
        completed = subprocess.run(args, capture_output=True, text=True, timeout=HTTP_TIMEOUT_SECONDS * 2)
    except subprocess.TimeoutExpired:
        result["result"] = "timeout"
        return result
    result["exit_code"] = completed.returncode
    if completed.returncode != 0:
        result["stderr"] = scrub(completed.stderr)
        return result
    try:
        payload = json.loads(completed.stdout or "null")
    except ValueError:
        result["result"] = "non-JSON output"
        return result
    budgets = (payload.get("budgets") or []) if isinstance(payload, dict) else (payload or [])
    result["budgets_returned"] = len(budgets)
    if isinstance(payload, dict):
        result["has_next_page"] = bool(payload.get("next_page_token"))
    return result


cli = shutil.which("databricks")
if IN_DATABRICKS_RUNTIME:
    print("Not executed: the CLI approach is for local use only.")
elif not (ACCOUNT_PROFILE and BUDGETS_PATH):
    print("Not executed: set DATABRICKS_ACCOUNT_PROFILE to an existing account-level CLI profile.")
elif not cli:
    print("Not executed: Databricks CLI not found on PATH.")
else:
    show(run_cli("databricks account budgets list", [cli, "account", "budgets", "list", "--profile", ACCOUNT_PROFILE, "--output", "json"]))
    show(run_cli("databricks api get /api/2.1/accounts/{account_id}/budgets", [cli, "api", "get", BUDGETS_PATH, "--profile", ACCOUNT_PROFILE]))

{
  "command": "databricks account budgets list",
  "exit_code": 0,
  "budgets_returned": 22
}


{
  "command": "databricks api get /api/2.1/accounts/{account_id}/budgets",
  "exit_code": 0,
  "budgets_returned": 22,
  "has_next_page": false
}


## Diagnostic: the same request with the workspace token

This is for comparison only. It sends the workspace credential from the control call to the account budgets endpoint, which is what you get if you assume a workspace login is enough. Expect it to fail.
- A `400` such as `Unable to load OAuth Config`, or a `401`, is an authentication or configuration failure. The response doesn't say why. One hypothesis is the documented rule that workspace tokens can't call account APIs.
- This failure says nothing about the user's account permissions, either way.

In [7]:
diagnostic_account_id = account_cfg.account_id if account_cfg is not None else (ACCOUNT_ID_OVERRIDE or w.config.account_id or "")
if not diagnostic_account_id:
    print("Not executed: no account_id available from DATABRICKS_ACCOUNT_ID, the account profile, or the workspace profile.")
else:
    diagnostic = requests.get(
        f"{ACCOUNTS_HOST}/api/2.1/accounts/{diagnostic_account_id}/budgets",
        headers=workspace_headers,
        timeout=HTTP_TIMEOUT_SECONDS,
    )
    show(summarize_http("GET /api/2.1/accounts/{account_id}/budgets (workspace token, diagnostic)", diagnostic, "budgets"))

{
  "call": "GET /api/2.1/accounts/{account_id}/budgets (workspace token, diagnostic)",
  "http_status": 400,
  "body_excerpt": "Unable to load OAuth Config"
}


## Reading the results

| Observation | Meaning |
|-------------|---------|
| Approaches 1 to 3 return HTTP 200 or exit code 0 with counts | The account-level credential is accepted and is authorized to list budgets. The counts reflect the account at run time. |
| Approach 1 and 2 counts differ from approach 3's `api get` | `api get` returns only the first page. Check `has_next_page`. |
| `403 PERMISSION_DENIED` on approaches 1 to 3 | The request authenticated and was then denied. The response doesn't name which role or permission is missing. |
| The account credential cell fails, or `cannot configure default credentials` appears | The SDK couldn't resolve an account credential. Possible causes include a missing or expired account login, or a wrong profile name. The error alone doesn't say which. |
| Diagnostic `400 Unable to load OAuth Config` or `401` | The account host rejected the workspace credential. This is an authentication or configuration failure, not a permissions result. |

Being in the workspace `admins` group (see the control output) doesn't by itself grant account-level access.